# Housing Approvals Forecaster — Full MLOps Pipeline Showcase

**Goal:** Forecast quarterly dwelling approvals per Australian LGA to track progress toward the *National Housing Accord* target of **1.2 million homes over 5 years**.

This notebook covers the project end-to-end:
- Pipeline architecture and feature engineering
- Model zoo: Seasonal Mean, SARIMA, LSTM — with honest comparison
- MLflow experiment tracking and model registry
- Champion model inference
- National Housing Accord progress tracker
- Monitoring system design

**Companion notebooks:**  
`01_eda.ipynb` — National trends, seasonal decomposition, LGA distribution  
`02_drift_case_study.ipynb` — 2022 rate-hike drift event in detail


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import mlflow
import pickle

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

FEATURES_PATH  = Path('../data/processed/features.parquet')
MLFLOW_DB      = Path('../mlflow.db').resolve()
SCALERS_PATH   = Path('../data/processed/scalers.pkl')
PREDICTIONS_DB = Path('../data/predictions.db')

mlflow.set_tracking_uri(f'sqlite:///{MLFLOW_DB}')
print('Setup complete.')

## 1. End-to-End Pipeline Architecture


In [ ]:
BLUE   = '#2C5F8A'
ORANGE = '#C86A1A'
GREEN  = '#2A7A4B'

def _box(ax, cx, cy, w, h, label, color, fs=9):
    rect = mpatches.FancyBboxPatch(
        (cx - w/2, cy - h/2), w, h,
        boxstyle='round,pad=0.015',
        facecolor=color, edgecolor='white', linewidth=1.5, zorder=3
    )
    ax.add_patch(rect)
    ax.text(cx, cy, label, ha='center', va='center',
            fontsize=fs, color='white', fontweight='bold', zorder=4)

def _arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5), zorder=2)

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# Row 1: data sources
_box(ax, 0.10, 0.82, 0.16, 0.10, 'ABS LGA2020\n2018-2020', BLUE)
_box(ax, 0.29, 0.82, 0.16, 0.10, 'ABS LGA2021\n2021-2025', BLUE)
_box(ax, 0.48, 0.82, 0.16, 0.10, 'RBA Cash Rate\nmonthly', BLUE)
_box(ax, 0.78, 0.82, 0.24, 0.10, 'data/pipeline.py\nfeatures.parquet\n3,371 rows x 16 cols', ORANGE)
_arrow(ax, 0.18, 0.82, 0.65, 0.82)
_arrow(ax, 0.37, 0.82, 0.65, 0.82)
_arrow(ax, 0.56, 0.82, 0.65, 0.82)

# Row 2: models
_box(ax, 0.18, 0.52, 0.20, 0.10, 'Seasonal Mean\nBaseline', ORANGE)
_box(ax, 0.50, 0.52, 0.20, 0.10, 'SARIMA\n(1,1,1)(0,1,1,4)', ORANGE)
_box(ax, 0.82, 0.52, 0.20, 0.10, 'HousingLSTM\n2-layer, h=64', ORANGE)
_arrow(ax, 0.78, 0.77, 0.18, 0.57)
_arrow(ax, 0.78, 0.77, 0.50, 0.57)
_arrow(ax, 0.78, 0.77, 0.82, 0.57)

# MLflow registry
_box(ax, 0.50, 0.34, 0.28, 0.10, 'MLflow Registry\n@champion alias', ORANGE)
_arrow(ax, 0.18, 0.47, 0.38, 0.37)
_arrow(ax, 0.50, 0.47, 0.50, 0.39)
_arrow(ax, 0.82, 0.47, 0.63, 0.37)

# Row 3: serving + monitoring
_box(ax, 0.18, 0.12, 0.24, 0.10, 'FastAPI\n/forecast  /health', GREEN)
_box(ax, 0.50, 0.12, 0.18, 0.10, 'SQLite\nprediction log', GREEN)
_box(ax, 0.82, 0.12, 0.24, 0.10, 'monitoring/\ndrift.py', GREEN)
_arrow(ax, 0.50, 0.29, 0.25, 0.17)
_arrow(ax, 0.30, 0.12, 0.41, 0.12)
_arrow(ax, 0.59, 0.12, 0.70, 0.12)

# Legend
legend_patches = [
    mpatches.Patch(facecolor=BLUE,   label='Data / Storage'),
    mpatches.Patch(facecolor=ORANGE, label='Processing / Models'),
    mpatches.Patch(facecolor=GREEN,  label='Serving / Monitoring'),
]
ax.legend(handles=legend_patches, loc='lower left', fontsize=10, framealpha=0.9)
ax.set_title('Housing Approvals Forecaster: End-to-End Pipeline', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

## 2. Data Pipeline: Feature Engineering

Two ABS sources (LGA2020: FY2017-18→2019-20, LGA2021: FY2020-21→2024-25) are consolidated with the RBA monthly cash rate into a single `features.parquet`. Each observation is one LGA × financial year, mapped to Q2 of the end year.

**11 modelling features:** `approvals_lag1`, `cash_rate`, `cash_rate_lag1/2`, `construction_cost_yoy`, `population_growth_yoy`, `season_q1–4`, `post_rate_hike`.


In [ ]:
df = pd.read_parquet(FEATURES_PATH)
df['quarter'] = df['quarter'].dt.to_period('Q')

from data.pipeline import describe
describe(df)

In [ ]:
CORR_COLS = [
    'dwellings_approved', 'approvals_lag1', 'approvals_yoy',
    'cash_rate', 'cash_rate_lag1', 'cash_rate_lag2',
    'construction_cost_yoy', 'population_growth_yoy',
    'season_q1', 'season_q2', 'season_q3', 'season_q4', 'post_rate_hike',
]
corr = df[CORR_COLS].corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label='Pearson r')

labels = [c.replace('_', '\n') for c in CORR_COLS]
ax.set_xticks(range(len(CORR_COLS)))
ax.set_yticks(range(len(CORR_COLS)))
ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
ax.set_yticklabels(labels, fontsize=8)

for i in range(len(CORR_COLS)):
    for j in range(len(CORR_COLS)):
        val = corr.values[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

ax.set_title('Pearson Correlation Matrix: Features vs Target', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

top3 = corr['dwellings_approved'].drop('dwellings_approved').abs().nlargest(3)
print('Top 3 correlates with dwellings_approved:')
print(top3.to_string())

## 3. Model Zoo

| Model | Description | Evaluation setup |
|---|---|---|
| **SeasonalMeanBaseline** | Rolling mean of same quarter over last 3 years per LGA. No free params. | Temporal split: train ≤ 2022Q2, test > 2022Q2 |
| **SARIMABaseline** | SARIMA(1,1,1)(0,1,1,4) per LGA, fitted with statsmodels. | Same temporal split |
| **HousingLSTM** | 2-layer LSTM, hidden=64, dropout=0.2, AdamW, CosineAnnealingLR, early stopping. | LGA-based 80/20 cross-sectional split |

> ⚠️ **Evaluation setup asymmetry:** Baselines use a temporal out-of-sample split that spans the 2022 structural break. The LSTM uses an LGA-based random split with no temporal hold-out. These evaluate different questions and the metrics are not directly comparable.


In [ ]:
from models.lstm import HousingLSTM

model_arch = HousingLSTM(input_size=11, hidden_size=64, num_layers=2, output_steps=1)
total_params = sum(p.numel() for p in model_arch.parameters())

print('HousingLSTM Architecture')
print('=' * 42)
print(f'  Input features   : 11')
print(f'  Hidden size      : 64')
print(f'  LSTM layers      : 2')
print(f'  Dropout          : 0.2')
print(f'  Output steps     : 1 (training)  /  4 (serving)')
print(f'  Sequence length  : 1 (training)  / 12 (serving)')
print('-' * 42)
print(f'  Total parameters : {total_params:,}')
print('=' * 42)
print(f'  Optimizer  : AdamW (lr=1e-3, wd=1e-4)')
print(f'  Scheduler  : CosineAnnealingLR')
print(f'  Loss       : MSELoss')
print(f'  Early stop : patience=10')

## 4. MLflow: Experiment Tracking and Model Registry

Every training run logs: hyperparameters, per-epoch `train_loss` and `val_mae`, final metrics, the PyTorch model artefact, and `scalers.pkl`. The model registry uses a `@champion` alias — the comparison script promotes the winner automatically.


In [ ]:
client = mlflow.tracking.MlflowClient()
exps = client.search_experiments()
print(f'Found {len(exps)} experiment(s)\n')

for exp in sorted(exps, key=lambda e: e.experiment_id):
    runs = client.search_runs([exp.experiment_id], order_by=['start_time DESC'], max_results=4)
    print(f'Experiment: {exp.name}  (id={exp.experiment_id})')
    for run in runs:
        m = run.data.metrics
        parts = [f'{k}={v:.3f}' for k, v in sorted(m.items()) if 'final' in k or 'mae' in k or 'mape' in k]
        status = run.info.status
        name = run.info.run_name or run.info.run_id[:8]
        print(f'  [{status:8}] {name:12}  {"  ".join(parts[:3])}')
    print()

In [ ]:
# Training curves from the best LSTM run
try:
    lstm_exp = next((e for e in exps if 'lstm' in e.name.lower()), None)
    if lstm_exp is None:
        raise RuntimeError('LSTM experiment not found')

    runs = client.search_runs(
        [lstm_exp.experiment_id],
        order_by=['metrics.final_val_mae ASC'],
        max_results=1,
    )
    run_id = runs[0].info.run_id
    best_epoch = int(runs[0].data.metrics.get('best_epoch', 0))
    print(f'Best LSTM run: {run_id[:8]}  '
          f'MAE={runs[0].data.metrics.get("final_val_mae", 0):.1f}  '
          f'best_epoch={best_epoch}')

    val_hist   = sorted(client.get_metric_history(run_id, 'val_mae'),   key=lambda x: x.step)
    train_hist = sorted(client.get_metric_history(run_id, 'train_loss'), key=lambda x: x.step)
    val_maes     = [m.value for m in val_hist]
    train_losses = [m.value for m in train_hist]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(val_maes, color='steelblue', linewidth=2)
    ax1.axvline(best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Best epoch {best_epoch}')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Val MAE (scaled)')
    ax1.set_title('Validation MAE')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(train_losses, color='darkorange', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Train MSE Loss (scaled)')
    ax2.set_title('Training Loss')
    ax2.grid(alpha=0.3)

    plt.suptitle(f'LSTM Training Curves  |  Run {run_id[:8]}', fontsize=12)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f'Could not load training curves: {e}')

In [ ]:
MODEL_NAME = 'housing-forecast-lstm'
try:
    versions = client.search_model_versions('name=' + repr(MODEL_NAME))
    print(f'Registered versions of "{MODEL_NAME}":')
    print(f'{"Ver":>4}  {"Run ID":>10}  {"Aliases":>20}')
    print('-' * 40)
    for v in sorted(versions, key=lambda x: int(x.version)):
        aliases = getattr(v, 'aliases', [])
        print(f'{v.version:>4}  {v.run_id[:8]:>10}  {str(aliases):>20}')
except Exception as e:
    print(f'Registry lookup failed: {e}')

## 5. Model Comparison: Who Won and Why

| Model | MAE (dwellings) | MAPE (fraction) | Hit @ ±15% |
|---|---|---|---|
| Seasonal Mean | **101.9** | **0.548** | **28.0%** |
| SARIMA | 163.0 | 0.834 | 21.2% |
| LSTM (best run) | 200.0 | 6.638 | 12.0% |

**Why did the baseline win?**
1. **Evaluation asymmetry** — Baselines are tested on quarters *after* the 2022 structural break, which is harder. LSTM uses a random LGA split without temporal hold-out.
2. **Dominant signal** — `approvals_lag1` explains most variance (see heatmap above). The seasonal mean implicitly uses this signal in two lines of code.
3. **LSTM trained cross-sectionally** — `seq_len=1` means each training sample is a single row; the LSTM never sees a temporal sequence during training. Its recurrent strength is unused.
4. **Small dataset** — ~500 LGAs × 7 years = 3,371 rows. Deep learning typically needs more data to beat well-chosen baselines.


In [ ]:
metrics_df = pd.DataFrame({
    'Model':    ['Seasonal Mean', 'SARIMA', 'LSTM'],
    'MAE':      [101.9, 163.0, 200.0],
    'MAPE':     [0.548, 0.834, 6.638],
    'Hit15':    [28.04, 21.24, 11.95],
})
COLORS = ['#2196F3', '#FF9800', '#F44336']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, xlabel, note in zip(
    axes,
    ['MAE', 'MAPE', 'Hit15'],
    ['MAE (dwellings / quarter)', 'MAPE (sklearn fraction)', 'Hit rate within +-15% (%)'],
    ['lower is better', 'lower is better', 'higher is better'],
):
    bars = ax.barh(metrics_df['Model'], metrics_df[col], color=COLORS,
                   edgecolor='white', height=0.5)
    ax.bar_label(bars, fmt='%.2f', padding=4, fontsize=10)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_xlim(0, metrics_df[col].max() * 1.25)
    ax.set_title(f'({note})', fontsize=9, color='gray', style='italic')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Model Comparison: MAE / MAPE / Hit Rate', fontsize=13)
fig.text(
    0.5, -0.04,
    'Note: Seasonal Mean and SARIMA evaluated on temporal test set (>2022Q2).  '
    'LSTM evaluated on LGA-based 20% holdout.  Results are not directly comparable.',
    ha='center', fontsize=9, color='gray', style='italic'
)
plt.tight_layout()
plt.show()

In [ ]:
from models.baseline import SeasonalMeanBaseline

TRAIN_END = pd.Period('2022Q2', freq='Q')
all_true, all_pred = [], []

for lga in df['lga_code'].unique():
    lga_df = df[df['lga_code'] == lga].sort_values('quarter')
    mask_tr = lga_df['quarter'] <= TRAIN_END
    mask_te = lga_df['quarter'] > TRAIN_END
    y_tr = lga_df[mask_tr]['dwellings_approved']
    y_te = lga_df[mask_te]['dwellings_approved']
    if len(y_tr) < 2 or len(y_te) == 0:
        continue
    y_tr_s = pd.Series(y_tr.values,
                       index=pd.PeriodIndex(lga_df[mask_tr]['quarter'].values, freq='Q'))
    sm = SeasonalMeanBaseline(n_years=3).fit(y_tr_s)
    preds = sm.predict(len(y_te))
    all_true.extend(y_te.values)
    all_pred.extend(preds[:len(y_te)])

all_true = np.array(all_true, dtype=float)
all_pred = np.array(all_pred, dtype=float)
clip_val = float(np.percentile(all_true, 98))

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(all_true, all_pred, alpha=0.25, s=15, color='steelblue', label='LGA x quarter')
lim = clip_val
ax.plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='1:1 perfect')
ax.plot([0, lim], [0, lim * 1.15], color='gray', linestyle=':', linewidth=1, alpha=0.6)
ax.plot([0, lim], [0, lim * 0.85], color='gray', linestyle=':', linewidth=1, alpha=0.6,
        label='+-15% band')
ax.set_xlim(0, clip_val)
ax.set_ylim(0, np.percentile(all_pred, 99) * 1.3)
ax.set_xlabel('Actual dwellings approved')
ax.set_ylabel('Predicted dwellings approved')
ax.set_title('Seasonal Mean: Predicted vs Actual (test set, post-2022Q2)')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

test_mae = np.mean(np.abs(all_true - all_pred))
test_hit = np.mean(np.abs(all_true - all_pred) / (np.abs(all_true) + 1e-8) <= 0.15)
print(f'Test MAE: {test_mae:.1f}  |  Hit@15%: {test_hit:.1%}')

In [ ]:
# 4-panel LGA-level forecast examples
lga_means = df.groupby('lga_code')['dwellings_approved'].mean().sort_values()
sample_lgas = [lga_means.index[int(len(lga_means) * q)] for q in [0.10, 0.35, 0.65, 0.90]]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, lga in zip(axes.flat, sample_lgas):
    lga_df = df[df['lga_code'] == lga].sort_values('quarter')
    mask_tr = lga_df['quarter'] <= TRAIN_END
    mask_te = lga_df['quarter'] > TRAIN_END
    y_tr = lga_df[mask_tr]['dwellings_approved']
    y_te = lga_df[mask_te]['dwellings_approved']
    mean_val = lga_df['dwellings_approved'].mean()

    xs_tr = [q.to_timestamp() for q in lga_df[mask_tr]['quarter']]
    xs_te = [q.to_timestamp() for q in lga_df[mask_te]['quarter']]

    if len(y_tr) < 2 or len(y_te) == 0:
        ax.text(0.5, 0.5, f'LGA {lga}\nInsufficient data',
                ha='center', va='center', transform=ax.transAxes)
        continue

    y_tr_s = pd.Series(y_tr.values,
                       index=pd.PeriodIndex(lga_df[mask_tr]['quarter'].values, freq='Q'))
    sm = SeasonalMeanBaseline(n_years=3).fit(y_tr_s)
    preds = sm.predict(len(y_te))

    ax.plot(xs_tr, y_tr.values, 'o-', color='steelblue',
            label='Train actuals', linewidth=2, markersize=5)
    ax.plot(xs_te, y_te.values, 'o-', color='darkorange',
            label='Test actuals', linewidth=2, markersize=5)
    ax.plot(xs_te, preds[:len(y_te)], 's--', color='green',
            label='SM prediction', linewidth=2, markersize=5)
    ax.axvline(pd.Timestamp('2022-07-01'), color='red',
               linestyle=':', alpha=0.7, linewidth=1.5, label='Q3 2022 hike')
    ax.set_title(f'LGA {lga}  (mean={mean_val:.0f} dwellings/yr)', fontsize=10)
    ax.set_ylabel('Dwellings approved')
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('LGA-Level Forecast Examples: Seasonal Mean vs Actuals', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Champion Model Inference

The serving stack: `load_champion()` fetches the `@champion` alias from the MLflow registry → scalers are loaded from `data/processed/scalers.pkl` → feature vector is tiled across a 12-step sequence window → PyTorch model produces a 4-quarter forecast → result is logged to SQLite.


In [ ]:
CHAMP_FEATURE_COLS = [
    'approvals_lag1', 'cash_rate', 'cash_rate_lag1', 'cash_rate_lag2',
    'construction_cost_yoy', 'population_growth_yoy',
    'season_q1', 'season_q2', 'season_q3', 'season_q4', 'post_rate_hike',
]
SEQ_LEN = 12

try:
    from models.registry import load_champion
    champ = load_champion('housing-forecast-lstm')
    champ.eval()

    with open(SCALERS_PATH, 'rb') as f:
        sc = pickle.load(f)
    x_sc, y_sc = sc['x_scaler'], sc['y_scaler']

    df_te = df[df['quarter'] > TRAIN_END].copy()
    sample_lgas = df_te['lga_code'].value_counts().head(6).index.tolist()

    print(f'{"LGA":>12}  {"Quarter":>8}  {"Actual":>8}  {"Predicted":>10}  {"Error":>8}')
    print('-' * 56)
    for lga in sample_lgas:
        row = df_te[df_te['lga_code'] == lga].sort_values('quarter').iloc[-1]
        feats = row[CHAMP_FEATURE_COLS].values.astype(np.float32)
        feats_sc = x_sc.transform(feats.reshape(1, -1))
        x_seq = torch.tensor(
            np.tile(feats_sc, (SEQ_LEN, 1))[np.newaxis], dtype=torch.float32
        )
        with torch.no_grad():
            pred_sc = champ(x_seq).cpu().numpy().ravel()
        pred = float(y_sc.inverse_transform(pred_sc.reshape(-1, 1)).ravel()[0])
        actual = int(row['dwellings_approved'])
        err = pred - actual
        print(f'{lga:>12}  {str(row["quarter"]):>8}  {actual:>8}  {pred:>10.1f}  {err:>+8.1f}')

except Exception as e:
    print(f'Champion model unavailable: {e}')
    print('Run: uv run python -m training.train_lstm')
    print('Then: uv run python -m training.compare')

**Equivalent API call** (server must be running on port 8000):

```bash
curl -X POST http://localhost:8000/forecast \
  -H 'Content-Type: application/json' \
  -d '{
    "lga_code": "24600",
    "features": {
      "approvals_lag1": 320, "cash_rate": 4.35,
      "cash_rate_lag1": 4.35, "cash_rate_lag2": 3.85,
      "construction_cost_yoy": 0.0, "population_growth_yoy": 0.0,
      "season_q1": 0, "season_q2": 1, "season_q3": 0, "season_q4": 0,
      "post_rate_hike": 1
    }
  }'
```

Response: `{lga_code, horizon_quarters: 4, predicted_approvals: [...], model_version, confidence_note}`


## 7. National Housing Accord Progress

The Accord target is **1.2 million homes over 5 years** (20 quarters from Q3 2022) = 60,000 dwellings/quarter nationally. The chart below tracks actual approvals against that pace using real data from `features.parquet`.


In [ ]:
ACCORD_START = pd.Period('2022Q3', freq='Q')
national = df.groupby('quarter')['dwellings_approved'].sum().reset_index()
national = national.sort_values('quarter')

post = national[national['quarter'] >= ACCORD_START].copy().reset_index(drop=True)
post['cumulative'] = post['dwellings_approved'].cumsum()

n = len(post)
TARGET_Q = 60_000
target_traj = np.arange(1, n + 1) * TARGET_Q
xticks = [str(q) for q in post['quarter']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.bar(range(n), post['dwellings_approved'], color='steelblue', alpha=0.85,
        label='Actual quarterly approvals')
ax1.axhline(TARGET_Q, color='red', linestyle='--', linewidth=2,
            label=f'Accord rate ({TARGET_Q:,}/quarter)')
ax1.set_xticks(range(n))
ax1.set_xticklabels(xticks, rotation=45, fontsize=9)
ax1.set_ylabel('Dwellings approved (national)')
ax1.set_title('Quarterly Approvals vs Accord Rate')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

ax2.plot(range(n), post['cumulative'], 'o-', color='steelblue',
         linewidth=2, label='Cumulative actuals')
ax2.plot(range(n), target_traj, 'r--', linewidth=2,
         label=f'Target trajectory (1.2M / 20Q)')
ax2.fill_between(
    range(n), post['cumulative'], target_traj,
    where=(target_traj > post['cumulative'].values),
    alpha=0.15, color='red', label='Supply gap'
)
ax2.set_xticks(range(n))
ax2.set_xticklabels(xticks, rotation=45, fontsize=9)
ax2.set_ylabel('Cumulative dwellings (national)')
ax2.set_title('Cumulative Progress vs Accord Target')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

gap = int(target_traj[-1] - post['cumulative'].iloc[-1])
plt.suptitle(
    f'National Housing Accord Progress  |  Current supply gap: {gap:,} dwellings',
    fontsize=13
)
plt.tight_layout()
plt.show()

## 8. Monitoring System

The monitoring stack runs two complementary drift signals:

| Component | Module | Trigger | GT needed? | Latency |
|---|---|---|---|---|
| PredictionLogger | `serving/logger.py` | Every `/forecast` call | No | Immediate |
| `detect_feature_drift` | `monitoring/drift.py` | Cash rate \|z\| > 2.5σ | No | Immediate |
| `detect_residual_drift` | `monitoring/drift.py` | Rolling MAE > 1.5× baseline | Yes (actuals) | 1–2 qtrs lag |
| `generate_drift_report` | `monitoring/drift.py` | Either signal active | Either | Composite |

**Key insight:** feature drift fires in the same quarter as the model-breaking event (cash rate hike Q3 2022). Residual drift follows 1–2 quarters later once ground truth accumulates.


In [ ]:
from monitoring.drift import _FEATURE_DRIFT_Z_THRESHOLD

rates = df.drop_duplicates('quarter').sort_values('quarter').copy()
train_cr = rates[rates['post_rate_hike'] == 0]['cash_rate'].dropna()
mean_cr, std_cr = train_cr.mean(), train_cr.std()
rates['z'] = (rates['cash_rate'] - mean_cr) / std_cr

xs = [q.to_timestamp() for q in rates['quarter']]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(xs, rates['z'], 'o-', color='steelblue', linewidth=2, markersize=6,
        label='Cash rate z-score')
ax.axhline(_FEATURE_DRIFT_Z_THRESHOLD, color='red', linestyle='--', linewidth=1.5,
           label=f'Drift threshold (+/-{_FEATURE_DRIFT_Z_THRESHOLD} sigma)')
ax.axhline(-_FEATURE_DRIFT_Z_THRESHOLD, color='red', linestyle='--', linewidth=1.5)
ax.axhline(0, color='green', linestyle=':', linewidth=1, alpha=0.6)
ax.axvline(pd.Timestamp('2022-07-01'), color='darkorange', linewidth=2, alpha=0.8,
           label='Q3 2022: First RBA rate hike')
ax.fill_between(xs, rates['z'], _FEATURE_DRIFT_Z_THRESHOLD,
                where=(rates['z'] > _FEATURE_DRIFT_Z_THRESHOLD),
                alpha=0.2, color='red', label='Drift zone')
ax.fill_between(xs, rates['z'], -_FEATURE_DRIFT_Z_THRESHOLD,
                where=(rates['z'] < -_FEATURE_DRIFT_Z_THRESHOLD),
                alpha=0.2, color='red')
ax.set_xlabel('Quarter')
ax.set_ylabel('Cash Rate Z-score vs Training Distribution')
ax.set_title('Feature Drift Signal: Cash Rate vs Pre-2022 Training Baseline')
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

current_z = float(rates['z'].iloc[-1])
current_cr = float(rates['cash_rate'].iloc[-1])
print(f'Training distribution: mean={mean_cr:.2f}%  std={std_cr:.2f}%')
print(f'Latest cash rate:      {current_cr:.2f}%  (z={current_z:.2f})')
triggered = 'DETECTED' if abs(current_z) > _FEATURE_DRIFT_Z_THRESHOLD else 'not triggered'
print(f'Feature drift:         {triggered}')

In [ ]:
from monitoring.drift import generate_drift_report

try:
    report = generate_drift_report(
        baseline_mae=101.9,
        window_days=90,
        current_cash_rate=current_cr,
        db_path=PREDICTIONS_DB,
        features_path=FEATURES_PATH,
    )
    print('Live Drift Report')
    print('=' * 40)
    print(f'  Baseline MAE:            {report.baseline_mae:.1f}')
    print(f'  Current rolling MAE:     {report.current_mae:.1f}  (n={report.n_predictions})')
    print(f'  MAE ratio:               {report.mae_ratio:.2f}x')
    print(f'  Feature drift detected:  {report.feature_drift_detected}')
    print(f'    Cash rate z-score:     {report.cash_rate_z_score}')
    print(f'  Residual drift detected: {report.residual_drift_detected}')
    print(f'    (requires logged predictions with actuals)')
except Exception as e:
    print(f'Drift report error: {e}')

## 9. Key Takeaways

**What this project demonstrates:**

1. **Full MLOps lifecycle** — Data ingestion (ABS + RBA APIs) → feature engineering → multi-model training → experiment tracking → model registry with `@champion` alias → typed FastAPI serving → prediction logging → drift monitoring. Each layer is independently testable.

2. **Honest benchmarking** — The seasonal mean baseline wins on all three metrics. The notebook explains why (evaluation asymmetry, dominant lag signal, cross-sectional LSTM, data volume). The `@champion` is set by automation but in practice the seasonal mean is the production-ready model.

3. **Two-tier drift detection** — Feature drift (cash rate z-score) fires immediately, without ground truth, in the same quarter as the structural break. Residual drift follows 1–2 quarters later once actuals are logged. The lag between signals is operationally useful: feature drift is the early warning, residual drift is the confirmation.

4. **Housing policy context** — A ±100 dwelling MAE per LGA per quarter matters differently for a 50-dwelling rural council than a 5,000-dwelling metro LGA. The Accord progress chart shows the national supply gap in plain terms.

---

## Notebook Navigation

| Notebook | Focus |
|---|---|
| `01_eda.ipynb` | National trends, seasonal decomposition, LGA distribution |
| `02_drift_case_study.ipynb` | 2022 rate-hike drift event: feature drift → residual drift timing |
| `03_project_showcase.ipynb` ← **this notebook** | Full MLOps pipeline, model comparison, champion inference |
